<a href="https://colab.research.google.com/github/paulheather147/FinalYearProject/blob/main/BaselineModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from datasets import load_dataset

import numpy as np

from sklearn.metrics import confusion_matrix, classification_report

img_size = 150
batch_size = 50
num_classes = 4

In [ ]:
dataset = load_dataset("Falah/Alzheimer_MRI")
print(dataset)
print(dataset["train"].features)

In [ ]:
train_val_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
train = train_val_split["train"]
val = train_val_split["test"]
test = dataset["test"]

def ensure_channel_dim(images):
    current_rank = tf.rank(images)

    def add_channel():
        return tf.expand_dims(images, axis=-1)

    def keep_same():
        return images

    return tf.cond(tf.equal(current_rank, 2), add_channel, keep_same)


def ensure_rgb_channels(images):
    current_channels = tf.shape(images)[-1]

    def to_rgb():
        return tf.image.grayscale_to_rgb(images)

    def drop_alpha():
        return images[..., :3]

    def keep_same():
        return images

    images = tf.cond(tf.equal(current_channels, 1), to_rgb, keep_same)

    new_channels = tf.shape(images)[-1]

    images = tf.cond(tf.equal(new_channels, 4), drop_alpha, keep_same)
    return images

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomFlip("horizontal"),
])

def simple_preprocess(x):
    return x / 255.0


def to_tensorflow_dataset(dataset_split, image_size, augment, preprocess_fn, shuffle=False):
    dataset_tf = dataset_split.to_tf_dataset(
        columns=["image", "label"],
        shuffle=shuffle,
        num_workers=0,
    )

    def preprocess_input(example_dict):
        images = tf.cast(example_dict["image"], tf.float32)
        images = ensure_channel_dim(images)
        images = ensure_rgb_channels(images)
        images.set_shape([None, None, 3])
        images = tf.image.resize(images, (image_size, image_size))

        if augment:
            images = data_augmentation(images)

        images = preprocess_fn(images)

        labels = tf.cast(example_dict["label"], tf.int32)
        labels = tf.one_hot(labels, depth=num_classes)
        return images, labels

    dataset_tf = dataset_tf.map(preprocess_input,
                                num_parallel_calls=tf.data.AUTOTUNE)
    dataset_tf = dataset_tf.batch(batch_size)
    return dataset_tf.prefetch(tf.data.AUTOTUNE)


train_ds = to_tensorflow_dataset(train, img_size, True, simple_preprocess, shuffle=True)
val_ds   = to_tensorflow_dataset(val,   img_size, False, simple_preprocess, shuffle=False)
test_ds  = to_tensorflow_dataset(test,  img_size, False, simple_preprocess, shuffle=False)

In [ ]:
model = models.Sequential([
    layers.Input(shape=(150, 150, 3)),

    layers.Conv2D(16, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(158, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])

model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 148, 148, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 41472)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 158)            │     6,552,734 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           636 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,558,458 (25.02 MB)

 Trainable params: 6,558,458 (25.02 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
opt = optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=opt,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

history = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[reduce_lr, early_stop]
)

test_loss, test_acc = model.evaluate(test_ds)
print("Test loss:", test_loss)
print("Test accuracy:", test_acc)

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)

    true_classes = np.argmax(labels.numpy(), axis=1)

    y_true.extend(true_classes)
    y_pred.extend(np.argmax(preds, axis=1))

print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred))


Epoch 1/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 25s 190ms/step - accuracy: 0.4218 - loss: 1.6664 - val_accuracy: 0.5020 - val_loss: 1.0867 - learning_rate: 0.0010
Epoch 2/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 114ms/step - accuracy: 0.5551 - loss: 0.9187 - val_accuracy: 0.5566 - val_loss: 0.9354 - learning_rate: 0.0010
Epoch 3/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 113ms/step - accuracy: 0.5966 - loss: 0.8628 - val_accuracy: 0.5928 - val_loss: 0.8673 - learning_rate: 0.0010
Epoch 4/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 115ms/step - accuracy: 0.6220 - loss: 0.8063 - val_accuracy: 0.6523 - val_loss: 0.7966 - learning_rate: 0.0010
Epoch 5/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 113ms/step - accuracy: 0.6771 - loss: 0.7189 - val_accuracy: 0.6914 - val_loss: 0.7090 - learning_rate: 0.0010
Epoch 6/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 114ms/step - accuracy: 0.7060 - loss: 0.6592 - val_accuracy: 0.7100 - val_loss: 0.6580 - learning_rate: 0.0010
Epoch 7/100
82/82 ━━━━━━━━━━━━━━━━━━━━ 9s 113ms/step - accuracy: 0.7502 - loss: 0